In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras.datasets as ds

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# [1] MNIST 데이터 불러오기
# - 흑백 손글씨 이미지: (28x28)
(x_train, y_train), (x_test, y_test) = ds.mnist.load_data()

# [2] Conv2D에 맞게 shape 추가: (28, 28, 1)
x_train = x_train.reshape(60000, 28, 28, 1)
x_test = x_test.reshape(10000, 28, 28, 1)

# [3] 픽셀값 정규화: 0~255 → 0~1
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

# [4] 레이블 One-hot 인코딩: 정수 → 벡터 (0~9)
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# [5] CNN 모델 정의 시작
cnn = Sequential()

# Conv Block 1: Conv → Conv → Pool → Dropout
cnn.add(Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
# - 첫 Conv: 32개의 3x3 필터로 에지/패턴 추출
cnn.add(Conv2D(32, (3, 3), activation='relu'))
# - 두 번째 Conv: 더 깊은 패턴 추출
cnn.add(MaxPooling2D(pool_size=(2, 2)))
# - 공간 해상도 절반으로 줄임 (28x28 → 14x14)
cnn.add(Dropout(0.25))
# - 일부 노드 랜덤 off → 과적합 방지

# Conv Block 2: Conv → Conv → Pool → Dropout
cnn.add(Conv2D(64, (3, 3), activation='relu'))
# - 필터 수 2배로 늘림: 더 복잡한 특징 학습
cnn.add(Conv2D(64, (3, 3), activation='relu'))
cnn.add(MaxPooling2D(pool_size=(2, 2)))
# - 다시 공간 축소 (14x14 → 7x7)
cnn.add(Dropout(0.25))

# Fully Connected 준비: Flatten
cnn.add(Flatten())
# - 3D 특징맵 → 1D 벡터

# Dense → Dropout
cnn.add(Dense(units=512, activation='relu'))
# - FC 층: 특징 조합 및 분류 준비
cnn.add(Dropout(0.5))
# - FC 계층은 과적합 위험이 높아 더 큰 드롭아웃

# 출력층: 10개 클래스, softmax로 확률 출력
cnn.add(Dense(units=10, activation='softmax'))

# [6] 모델 컴파일
cnn.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

# [7] 학습 수행
# - 배치크기 128, epoch 100
# - 검증 데이터: x_test, y_test
hist = cnn.fit(
    x_train, y_train,
    batch_size=128,
    epochs=100,
    validation_data=(x_test, y_test),
    verbose=2
)

# [8] 학습된 모델 저장
cnn.save('cnn_v2.h5')

# [9] 최종 테스트 정확도 평가
res = cnn.evaluate(x_test, y_test, verbose=0)
print('정확률 =', res[1] * 100)

print('정확률=',res[1]*100)

Epoch 1/100
469/469 - 6s - loss: 0.2248 - accuracy: 0.9280 - val_loss: 0.0443 - val_accuracy: 0.9844 - 6s/epoch - 13ms/step
Epoch 2/100
469/469 - 1s - loss: 0.0668 - accuracy: 0.9793 - val_loss: 0.0265 - val_accuracy: 0.9923 - 1s/epoch - 3ms/step
Epoch 3/100
469/469 - 1s - loss: 0.0480 - accuracy: 0.9854 - val_loss: 0.0206 - val_accuracy: 0.9931 - 1s/epoch - 3ms/step
Epoch 4/100
469/469 - 1s - loss: 0.0388 - accuracy: 0.9881 - val_loss: 0.0203 - val_accuracy: 0.9932 - 1s/epoch - 3ms/step
Epoch 5/100
469/469 - 1s - loss: 0.0331 - accuracy: 0.9898 - val_loss: 0.0216 - val_accuracy: 0.9934 - 1s/epoch - 3ms/step
Epoch 6/100
469/469 - 1s - loss: 0.0299 - accuracy: 0.9905 - val_loss: 0.0184 - val_accuracy: 0.9935 - 1s/epoch - 3ms/step
Epoch 7/100
469/469 - 1s - loss: 0.0252 - accuracy: 0.9920 - val_loss: 0.0186 - val_accuracy: 0.9939 - 1s/epoch - 3ms/step
Epoch 8/100
469/469 - 1s - loss: 0.0235 - accuracy: 0.9926 - val_loss: 0.0174 - val_accuracy: 0.9940 - 1s/epoch - 3ms/step
Epoch 9/100
469